# 03 - Sentiment Model Training (TF-IDF + Logistic Regression)

Trains the production sentiment classifier on a real reviews dataset (e.g. Kaggle 'Women's E-Commerce Clothing Reviews'), replacing the small seed-corpus cold-start model in `app/services/nlp_service.NLPService`.

In [10]:
import sys
sys.path.append('..')
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib
from app.services.nlp_service import NLPService

In [11]:
df = pd.read_csv('../data/reviews.csv')
df = df.rename(columns={'Review Text': 'review_text', 'Rating': 'rating'})
df = df.dropna(subset=['review_text', 'rating'])

def rating_to_label(r):
    if r <= 2: return 'Negative'
    if r == 3: return 'Neutral'
    return 'Positive'

df['label'] = df['rating'].apply(rating_to_label)
svc = NLPService()
df['cleaned'] = df['review_text'].astype(str).apply(svc.clean_text)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned'], df['label'], test_size=0.2, stratify=df['label'], random_state=42
)

vectorizeEr = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_vec, y_train)

print(classification_report(y_test, model.predict(X_test_vec)))

              precision    recall  f1-score   support

    Negative       0.51      0.59      0.55       474
     Neutral       0.38      0.51      0.44       565
    Positive       0.95      0.88      0.92      3490

    accuracy                           0.80      4529
   macro avg       0.62      0.66      0.63      4529
weighted avg       0.84      0.80      0.82      4529



In [12]:
joblib.dump(model, '../app/models/sentiment_model.pkl')
joblib.dump(vectorizer, '../app/models/vectorizer.pkl')
print('Saved sentiment_model.pkl and vectorizer.pkl to app/models/')

Saved sentiment_model.pkl and vectorizer.pkl to app/models/
